In [ ]:
# %%
# 🧠 EMG Signal Processing Workflow
# This notebook walks through a simple pipeline for electromyographic (EMG)
# data analysis, from loading raw recordings to extracting features and
# building a toy classifier. Each section is clearly marked so you can follow
# along step-by-step and adapt the code for your own datasets.

# -------- 1. Import Libraries --------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from scipy.fft import rfft, rfftfreq
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# set a nicer plotting style
plt.style.use('seaborn-darkgrid')

# -------- Helper functions --------
def bandpass_filter(signal, lowcut, highcut, fs, order=4):
    """Apply a Butterworth band‑pass filter to the input signal."""
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)


def median_frequency(signal, fs):
    """Compute the median frequency of a signal's power spectrum."""
    fft_vals = np.abs(rfft(signal))
    freqs = rfftfreq(len(signal), 1/fs)
    cumulative = np.cumsum(fft_vals)
    idx = np.where(cumulative >= cumulative[-1] / 2)[0][0]
    return freqs[idx]

In [ ]:
# %%
# -------- 2. Load data --------
# In Colab you can upload files; here we assume the CSV is in the working
# directory or has been uploaded manually.

# Uncomment the following if running in Google Colab:
# from google.colab import files
# uploaded = files.upload()

# Replace the filename with your dataset path as needed
filename = 'emg_data.csv'
try:
    data = pd.read_csv(filename)
    print(f"Loaded {len(data)} samples from {filename}")
except FileNotFoundError:
    print(f"File {filename} not found. Make sure it is in the current folder.")
    data = pd.DataFrame()

# Extract columns if they exist
if not data.empty:
    time = data['time'].values
    emg = data['emg'].values
else:
    time = np.array([])
    emg = np.array([])

# sampling frequency (Hz)
fs = 1000  # change if your acquisition system uses something else


In [ ]:
# %% [markdown]
## 3. Preprocessing

- **Band‑pass filter** to remove motion artifacts and high‑frequency noise (20–450 Hz).
- **Rectify** the signal to make all values positive.
- Compute **Root‑Mean‑Square (RMS) envelope** as a smooth measure of muscle activity.

In [ ]:
# %%
# only run processing if we have data
if emg.size > 0:
    filtered_emg = bandpass_filter(emg, lowcut=20, highcut=450, fs=fs)
    rectified_emg = np.abs(filtered_emg)

    # RMS envelope using a sliding window (samples)
    window = int(0.2 * fs)  # 200-ms window
    rms = np.sqrt(
        np.convolve(rectified_emg**2, np.ones(window)/window, mode='valid')
    )
else:
    filtered_emg = np.array([])
    rectified_emg = np.array([])
    rms = np.array([])


In [ ]:
# %% [markdown]
## 4. Feature Extraction

From the preprocessed signal we calculate:

1. **Mean and standard deviation of the RMS envelope** – gives an overall activity level.
2. **Median frequency** of the band‑pass filtered signal – often used to assess muscle fatigue.

In [ ]:
# %%
# compute features only if rms exists
if rms.size > 0:
    feat_mean = np.mean(rms)
    feat_std = np.std(rms)
    feat_mf = median_frequency(filtered_emg, fs)
    feature_vector = np.array([[feat_mean, feat_std, feat_mf]])
    print(f"Features: mean={feat_mean:.4f}, std={feat_std:.4f}, mf={feat_mf:.1f} Hz")
else:
    feature_vector = np.zeros((1,3))


In [ ]:
# %% [markdown]
## 5. Toy Classifier Example

We split a small synthetic dataset to demonstrate training a classifier. In practice
replace `X` and `y` with a larger labeled EMG dataset.

In [ ]:
# %%
# synthetic training data
X = np.array([
    [0.10, 0.02, 130],
    [0.12, 0.03, 120],
    [0.30, 0.08, 70],
    [0.28, 0.07, 75]
])
y = np.array(["healthy", "healthy", "impaired", "impaired"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
print(f"Training accuracy: {model.score(X_train, y_train):.2f}")
print(f"Test accuracy: {model.score(X_test, y_test):.2f}")

if rms.size > 0:
    prediction = model.predict(feature_vector)
    print("Predicted condition:", prediction[0])


In [ ]:
# %% [markdown]
## 6. Visualization Enhancements

We'll plot multiple stages of the signal in a single figure with clear labels
and color-coded lines. This makes it easier to compare raw, filtered, and
processed signals.

In [ ]:
# %%
if emg.size > 0:
    fig, axs = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
    axs[0].plot(time, emg, color='tab:blue', label='raw')
    axs[0].set(title='Raw EMG', ylabel='Amplitude')
    axs[0].legend()

    if filtered_emg.size > 0:
        axs[1].plot(time, filtered_emg, color='tab:orange', label='filtered')
        axs[1].set(title='Filtered EMG (20–450 Hz)', ylabel='Amplitude')
        axs[1].legend()

    if rectified_emg.size > 0:
        axs[2].plot(time, rectified_emg, color='tab:green', label='rectified')
        axs[2].set(title='Rectified EMG', ylabel='Amplitude')
        axs[2].legend()

    if rms.size > 0:
        rms_time = np.linspace(time[0], time[-1], len(rms))
        axs[3].plot(rms_time, rms, color='tab:red', label='RMS envelope')
        axs[3].set(title='RMS Envelope', xlabel='Time (s)', ylabel='RMS')
        axs[3].legend()

    plt.tight_layout()
    plt.show()

    # also plot the power spectrum with median frequency marker
    if filtered_emg.size > 0:
        fft_vals = np.abs(rfft(filtered_emg))
        freqs = rfftfreq(len(filtered_emg), 1/fs)
        plt.figure(figsize=(8,4))
        plt.plot(freqs, fft_vals, color='tab:purple')
        mf_val = median_frequency(filtered_emg, fs)
        plt.axvline(mf_val, color='k', linestyle='--', label=f'Median freq = {mf_val:.1f} Hz')
        plt.title('Power Spectrum of Filtered EMG')
        plt.xlabel('Frequency (Hz)')
        plt.ylabel('Magnitude')
        plt.legend()
        plt.xlim(0, 500)
        plt.show()
else:
    print("No data to visualize.")